In [1]:
import sys
sys.path.append('..')

from pathlib import Path
import pandas as pd
import joblib

from src.train import load_target
from src.budget_sim import (
    get_raw_monthly_charges, compute_customer_economics,
    profit_curve, find_optimal_k, plot_profit_curve,
    top_k_targeting_table, write_model_card,
    RETENTION_HORIZON_MONTHS, CONTACT_COST_PCT, SAVE_RATE,
)

DATA_DIR = Path("data/processed")
MODEL_DIR = Path("models")
REPORT_DIR = Path("reports")
FIG_DIR = REPORT_DIR / "figures"

X_test_encoded = pd.read_csv(DATA_DIR / "X_test_encoded.csv")
y_test = load_target(DATA_DIR / "y_test.csv")

# Calibrated model — this phase uses the PRODUCTION (calibrated) model,
# unlike Phase 4 which explained the pre-calibration model.
cal_model = joblib.load(MODEL_DIR / "lgbm_calibrated_sigmoid.pkl")

y_prob = cal_model.predict_proba(X_test_encoded)[:, 1]
print(f"Loaded {len(y_prob)} calibrated churn probabilities")

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/X_test_encoded.csv'

In [ ]:
monthly_charges = get_raw_monthly_charges(DATA_DIR / "X_test.csv", n_rows_expected=len(y_test))

print(f"Raw MonthlyCharges range: ${monthly_charges.min():.2f} - ${monthly_charges.max():.2f}")
print(f"Mean: ${monthly_charges.mean():.2f}")

In [ ]:
clv, contact_cost = compute_customer_economics(
    monthly_charges,
    retention_horizon_months=RETENTION_HORIZON_MONTHS,
    contact_cost_pct=CONTACT_COST_PCT,
)

print(f"CLV proxy range: ${clv.min():.2f} - ${clv.max():.2f}")
print(f"Contact cost range: ${contact_cost.min():.2f} - ${contact_cost.max():.2f}")

In [ ]:
optimal_row = find_optimal_k(curve_df)
print(optimal_row)

fig = plot_profit_curve(curve_df, optimal_row=optimal_row, out_path=FIG_DIR / "profit_curve.png")
fig

In [ ]:
optimal_k_frac = optimal_row["k_pct"] / 100
targeting_table = top_k_targeting_table(y_prob, monthly_charges, clv, contact_cost, k_frac=optimal_k_frac)

print(f"Top {len(targeting_table)} customers to contact:")
print(targeting_table.head(15).to_string(index=False))

targeting_table.to_csv(REPORT_DIR / "top_k_targeting_list.csv", index=False)
print("\nSaved full list: reports/top_k_targeting_list.csv")

In [ ]:
for alt_rate in [0.15, 0.30, 0.45, 0.60]:
    alt_curve = profit_curve(y_test, y_prob, clv, contact_cost, save_rate=alt_rate)
    alt_optimal = find_optimal_k(alt_curve)
    print(f"save_rate={alt_rate:.0%}: optimal k={alt_optimal['k_pct']:.0f}%, "
          f"profit=${alt_optimal['total_profit']:,.0f}")

In [ ]:
metrics_df = pd.read_csv(REPORT_DIR / "phase3_calibration_metrics.csv")
best_metrics = metrics_df.sort_values("brier_score").iloc[0]

# Update this list from your team's final, VERIFIED Phase 4 top-drivers table —
# including the resolved fiber-optic direction once that's closed out.
top_shap_drivers = [
    ("Contract", "increases churn risk"),
    ("tenure", "decreases churn risk"),
    ("InternetService_Fiber_optic", "see dependence plot — direction under review"),
    ("PaymentMethod_Electronic_check", "increases churn risk"),
    ("InternetService_No", "decreases churn risk"),
]

print(best_metrics)

In [ ]:
card_path = write_model_card(
    REPORT_DIR,
    brier_score=best_metrics["brier_score"],
    calibration_method=best_metrics["model"],
    roc_auc=best_metrics["roc_auc"],
    ece=best_metrics["ece"],
    optimal_k_row=optimal_row,
    top_shap_drivers=top_shap_drivers,
)

print(f"Saved {card_path}")

In [ ]:
print("=== Phase 5 Checkpoint (Project Complete) ===")
print(f"Optimal targeting depth: top {optimal_row['k_pct']:.0f}% ({int(optimal_row['n_contacted'])} customers)")
print(f"Expected profit: ${optimal_row['total_profit']:,.0f} (at save_rate={SAVE_RATE:.0%} — see Cell 7)")
print()
print("Files written:")
print("  - reports/model_card.md")
print("  - reports/top_k_targeting_list.csv")
print("  - reports/figures/profit_curve.png")
print()
print("3 deliverables complete:")
print("  1. Calibrated LightGBM, Brier < 0.14 -- PASSED (Phase 3)")
print("  2. SHAP churn-driver report -- Phase 4")
print("  3. Top-k targeting simulation + model card -- Phase 5")
print()
print("Next: resolve open Phase 4 verification items, get a real save_rate from")
print("the business side if available, git commit, present to the team.")